<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
daily_files = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
dim_content_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                    filename="dim_content.parquet", token=token)
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in daily_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

# Same grain, windows, and verified column set as w02/w03 -- nothing new invented here.
# Dropped: sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, sessions_referral,
# sessions_social, sessions_paid -- all measure traffic from a channel that has no
# plausible link to a GSC-impression-based label (this notebook's target is built
# entirely from search impressions, a different measurement system). Sparsity was
# a secondary factor; channel relevance to the actual label is the real reason.
# sessions_organic/direct kept: organic substantially overlaps the same search
# visibility GSC measures, direct plausibly tracks general site/brand strength.
query = f"""
WITH prior AS (
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,
        -- gsc_avg_position (daily) == gsc_sum_position / gsc_impressions (daily),
        -- verified exactly against real rows. So the correct 90-day position is an
        -- IMPRESSION-WEIGHTED average -- sum the numerator and denominator across
        -- days first, divide once -- not an unweighted average of 90 daily averages
        -- (which lets a single low-traffic day skew the result as much as a
        -- high-traffic one). Still excludes the sentinel-zero "no position data"
        -- days from both sides of the ratio, same as before.
        SUM(gsc_sum_position) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_sum,
        SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_impressions,
        -- Consistency-of-visibility features, matching the reference pipeline's
        -- days_with_impressions/days_with_sessions -- how many of the 90 days
        -- actually had activity, not just how much total activity occurred.
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_sessions > 0) AS days_with_sessions,
        BOOL_OR(ga4_data_available) AS ga4_data_available,
        SUM(ga4_pageviews) AS ga4_pageviews, SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users, SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic, SUM(sessions_direct) AS sessions_direct,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
              AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_baseline_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_recent_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
    -- label-window aggregate -- kept separate, joined only for the leakage
    -- demo in section 3, NEVER merged into the feature columns below.
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date <= DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
WHERE p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
"""
df = con.sql(query).df()

# Impression-weighted 90-day average position, computed once from the summed
# numerator/denominator -- NaN when no day in the window had real position data.
df["gsc_avg_position"] = df["pos_weighted_sum"] / df["pos_weighted_impressions"].replace(0, np.nan)

# dim_content join -- content metadata, verified 100% join rate in w03.
# Drop dim_content's own client_hash_id first -- df already has one from the
# daily fact table, and the two columns colliding breaks the next merge.
dim = con.sql(f"SELECT * FROM read_parquet('{dim_content_file}')").df()
dim = dim.drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

# dim_clients join -- for the coverage-gap flag (w03's 26% finding), not as a feature.
clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{clients_file}')").df()
df = df.merge(clients, on="client_hash_id", how="left")

print("Rows before any filtering:", len(df))

# --- Client-coverage filter: the w03/w02 caveat, actually applied now ---
# Drop rows whose client's gsc_data_start falls inside our 90-day prior window --
# their "zero" history is a coverage gap, not a real signal.
prior_window_start = pd.Timestamp(DECISION_DATE) - pd.Timedelta(days=90)
coverage_ok = df["gsc_data_start"].isna() | (df["gsc_data_start"] <= prior_window_start)
print("Dropped for incomplete client coverage:", (~coverage_ok).sum(),
      f"({(~coverage_ok).mean():.1%})")
df = df[coverage_ok].copy()
print("Rows after coverage filter:", len(df))
print()

# --- Prior-window trend (safe: entirely before the decision point) ---
df["prior_trend_pct"] = (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
df["was_declining"] = df["prior_trend_pct"] <= -20

# --- Content age / freshness -- present in dim_content but never converted until
# now. This mirrors content_age_days/days_since_last_update in the reference
# pipeline's MODEL_NUMERIC_FEATURES (scripts/ml_utils.py) -- a real omission,
# not a deferred decision (this is exactly what Week 2's hand rule used).
decision_ts = pd.Timestamp(DECISION_DATE)
df["content_age_days"] = (decision_ts - pd.to_datetime(df["content_created_date"])).dt.days
df["days_since_last_update"] = (decision_ts - pd.to_datetime(df["content_updated_date"])).dt.days

# --- Derived rates, matching the reference pipeline's ctr/engagement_rate/scroll_rate.
# Undefined only when the denominator is 0 (no visibility/sessions at all that
# window) -- exactly the same rows already flagged by has_position_data/has_ga4_data,
# so no new flag needed. 0 is a SAFE fill here (unlike avg_position): w03 verified
# 56.8% of tracked pages genuinely have 0 CTR despite real impressions, so a 0 fill
# blends into an already-common, legitimate value rather than faking an extreme.
df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan) * 100).fillna(0)
df["scroll_rate"] = (df["scroll_events"] / df["ga4_pageviews"].replace(0, np.nan) * 100).fillna(0)

# --- has_* flags BEFORE any fillna -- never blindly fill a real sentinel ---
df["has_position_data"] = df["gsc_avg_position"].notna().astype(int)
# ga4_data_available can itself be NA (the "undetermined" rows found in w03) --
# treat undetermined the same as "not confirmed available" for this flag.
df["has_ga4_data"] = df["ga4_data_available"].fillna(False).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_backlink_data"] = df["backlinks"].notna().astype(int)

# Now fill, with the flags already recording what was really missing.
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(df["gsc_avg_position"].median())
for col in ["search_volume", "competition", "cpc", "word_count", "char_count", "backlinks",
            "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
            "ga4_total_engagement_sec", "sessions_organic", "sessions_direct", "scroll_events"]:
    df[col] = df[col].fillna(0)
df["main_intent"] = df["main_intent"].fillna("unknown")
df["content_type"] = df["content_type"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

# --- log-transform the heavy-tailed counts ---
# gsc_sum_position and ga4_engaged_sessions were missed here in an earlier pass --
# both are just as heavy-tailed as the others (verified: gsc_sum_position reaches
# the hundreds of thousands, same scale as gsc_impressions; ga4_engaged_sessions
# maxes at 406 with a long tail of small values) -- log every count consistently,
# not a hand-picked subset.
for col in ["gsc_impressions", "gsc_clicks", "ga4_sessions", "search_volume", "backlinks",
            "scroll_events", "gsc_sum_position", "ga4_engaged_sessions"]:
    df[f"log_{col}"] = np.log1p(df[col])

print("Feature vector shape:", df.shape)
df.head()

Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows before any filtering: 134398
Dropped for incomplete client coverage: 18652 (13.9%)


Rows after coverage filter: 115746



Feature vector shape: (115746, 67)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,pos_weighted_sum,pos_weighted_impressions,days_with_impressions,days_with_sessions,ga4_data_available,...,has_word_count,has_backlink_data,log_gsc_impressions,log_gsc_clicks,log_ga4_sessions,log_search_volume,log_backlinks,log_scroll_events,log_gsc_sum_position,log_ga4_engaged_sessions
0,content_8d54ba68c975c904,client_62f4a7e64f5e0096,39128.0,59.0,176367.0,176367.0,39128.0,90,0,False,...,1,1,10.574619,4.094345,0.0,2.397895,4.718499,0.0,12.080328,0.0
1,content_76b567d085fe768d,client_62f4a7e64f5e0096,54914.0,187.0,245299.0,245299.0,54914.0,90,0,False,...,1,1,10.913542,5.236442,0.0,2.397895,0.0,0.0,12.410237,0.0
2,content_b4219132ed66f5f9,client_62f4a7e64f5e0096,99.0,0.0,2088.0,2088.0,96.0,43,0,False,...,0,1,4.605170,0.000000,0.0,2.397895,0.0,0.0,7.644441,0.0
3,content_4cf5771a50c921fe,client_62f4a7e64f5e0096,30511.0,79.0,113590.0,113590.0,30511.0,90,0,False,...,1,1,10.325875,4.382027,0.0,2.397895,0.0,0.0,11.640360,0.0
4,content_80f27426898c539d,client_62f4a7e64f5e0096,6672.0,5.0,9821.0,9821.0,6672.0,90,0,False,...,1,1,8.805825,1.791759,0.0,0.0,0.0,0.0,9.192380,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision point? |
|---|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | 90-day GSC totals — additive, so a straight `SUM` over the window is correct | Zero-filled; `gsc_sum_position` is kept raw mainly as the numerator behind `gsc_avg_position` below | Yes — entire 90-day prior window |
| `gsc_avg_position` | **Impression-weighted** 90-day average position: `SUM(gsc_sum_position) / SUM(gsc_impressions)`, both filtered to days with real position data. NOT an unweighted average of 90 daily averages — verified that `gsc_avg_position` (daily) = `gsc_sum_position / gsc_impressions` (daily) exactly, so combining days requires summing numerator and denominator first, or a single very low-traffic day would skew the result as much as a high-traffic one. | Kept `NaN` until flagged (`has_position_data`), then median-filled — never a blind zero (verified sentinel trap in `w01`/`w02`) | Yes — entire 90-day prior window |
| `ctr` | `gsc_clicks / gsc_impressions × 100`, matching the reference pipeline's own `ctr` feature (`scripts/ml_utils.py`) | Undefined only when `gsc_impressions = 0` — the same rows already covered by `has_position_data`. `0` is a **safe** fill here (unlike `avg_position`): `w03` verified 56.8% of tracked pages genuinely have 0 CTR despite real impressions, so filling `0` blends into an already-common, legitimate value rather than faking an extreme. | Yes |
| `days_with_impressions`, `days_with_sessions` | Count of the 90 days that had ≥1 impression / ≥1 GA4 session — a *consistency* signal (steady low traffic vs. one spike), not just total volume. Matches the reference pipeline's own features. | Not missing by construction (a `COUNT` over the window, 0 is a real, valid count) | Yes |
| `content_age_days`, `days_since_last_update` | Days between the decision point and `content_created_date`/`content_updated_date` (`dim_content`) — the exact features Week 2's hand rule (`stale x visible`) used. A real omission in earlier drafts, not a deferred decision. | `content_created_date`/`content_updated_date` are 0% missing in `dim_content` (verified) — no fill needed | Yes — both dates are always in the past relative to any reasonable decision point |
| `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `scroll_events` | 90-day GA4 totals — kept because organic/direct sessions and on-page engagement/scroll plausibly connect to the same search-visibility story the label measures | Zero-filled, but only trustworthy alongside `has_ga4_data` (verified: 51.7% of items never tracked at all, 19.5% undetermined — `w03`) | Yes — same window |
| `engagement_rate`, `scroll_rate` | `ga4_engaged_sessions / ga4_sessions × 100`, `scroll_events / ga4_pageviews × 100` — matching the reference pipeline | Undefined only when the GA4 denominator is 0 — already covered by `has_ga4_data`. `0` fill is safe for the same reason as `ctr`. | Yes |
| `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` | Keyword-context metadata from `dim_content` | `has_keyword_data` flag, then zero/`"unknown"` fill (18.3-18.5% missing, matches "no keyword data" pattern) | Yes — static/slow-changing metadata |
| `word_count`, `char_count` | Content properties | `has_word_count` flag, then zero fill (30.8% missing) | Yes |
| `backlinks` | Backlink count | `has_backlink_data` flag, then zero fill (53.0% missing) | Yes |
| `content_type`, `category_count` | Content metadata | `content_type` filled `"unknown"`; `category_count` fully populated (0% missing) | Yes |
| `prior_trend_pct`, `was_declining` | This notebook's own trend-check computation | Not missing by construction (rows without valid trend data are excluded at the query stage) | Yes — computed entirely from the 30-vs-30 trend-check window, strictly before the decision point |
| `log_gsc_impressions`, `log_gsc_clicks`, `log_ga4_sessions`, `log_search_volume`, `log_backlinks`, `log_scroll_events`, `log_gsc_sum_position`, `log_ga4_engaged_sessions` | `log1p` of every heavy-tailed raw count — applied consistently to all of them, not a hand-picked subset. `gsc_sum_position` and `ga4_engaged_sessions` were missed in an earlier pass (used raw) despite being just as heavy-tailed as the rest; fixed. | Same as the raw column | Yes |

**Log + scale, why both, and why in that order:** `log1p` fixes *shape* (a single page with ~800K impressions would otherwise dominate a linear model). `StandardScaler` (applied at model-fit time in section 3, fit on train only — never baked into this stored feature vector) fixes *scale* (putting `log_gsc_impressions`, `word_count`, and a 0/1 flag onto comparable units). They solve different problems and don't undo each other — `StandardScaler` is a linear shift-and-rescale, so it preserves whatever shape `log1p` already fixed. Order matters: log first (needs non-negative input), scale second (reversing would try to take `log()` of negative, mean-centered values).

**Dropped: `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `sessions_referral`, `sessions_social`, `sessions_paid`.** These measure traffic from channels (AI referral, paid, social, other-site referral) with no plausible link to a *GSC-impression-based* label — this notebook's target is built entirely from search impressions, a different measurement system than any of these. Sparsity (85-99.7% zero) was a secondary factor; channel relevance to the actual target was the deciding one. `ai_traffic_pct` (a reference-pipeline feature) is left out for the same reason. See section 4.

**Sum vs. average, the general rule:** additive quantities over the window (impressions, clicks, sessions, engagement seconds — total activity that occurred) get `SUM`. Rate/characteristic quantities describing a *quality* per day (like average position) need proper combining, not just summing OR naive daily-averaging — verify what the underlying columns actually relate to (here, `sum_position / impressions`) before picking either.

**Categorical fields** (`main_intent`, `content_type`, `competition_level`): one-hot/ordinal encoding is a modeling-stage decision (ML-08), not done here — this notebook just guarantees they're clean, correctly-typed strings with no leaked info.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
%pip install -q scikit-learn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Labels: same recent-30-day baseline established (and tested) in w02.
recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100
df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)

# Channel-relevant candidates, now including the reference pipeline's own
# MODEL_NUMERIC_FEATURES that were missing here until this pass: content_age_days,
# days_since_last_update, ctr, engagement_rate, scroll_rate, days_with_impressions,
# days_with_sessions (scripts/ml_utils.py). ai_traffic_pct is the one reference
# feature deliberately left out -- consistent with dropping the other AI/channel
# columns above, no plausible link to a GSC-impression-based label.
# gsc_sum_position/ga4_engaged_sessions use their log_ versions, not raw -- both
# are heavy-tailed just like the other logged counts; using them raw here was an
# earlier inconsistency, now fixed.
honest_features = [
    "gsc_avg_position", "log_gsc_sum_position", "prior_trend_pct",
    "log_gsc_impressions", "log_gsc_clicks", "log_ga4_sessions", "log_search_volume", "log_backlinks",
    "log_scroll_events", "log_ga4_engaged_sessions", "word_count", "char_count", "category_count",
    "content_age_days", "days_since_last_update", "ctr", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions",
    "has_position_data", "has_ga4_data", "has_keyword_data", "has_word_count", "has_backlink_data",
]
X = df[honest_features].fillna(0)
y = df["future_decline"].astype(int)
groups = df["client_hash_id"]

# Unscaled features (raw counts next to 0/1 flags) caused convergence failures
# and an unreliable "honest" number -- scale before fitting, every time.
def fit_scaled(X_train, y_train, X_test):
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000)
    model.fit(scaler.transform(X_train), y_train)
    return model.predict_proba(scaler.transform(X_test))[:, 1]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

honest_probs = fit_scaled(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probs)
print(f"Honest features, grouped split -- test AUC: {honest_auc:.3f}")
print(f"Base rate (future_decline): {y.mean():.1%} -- this is the real, scaled, converged")
print("baseline the honest feature set earns. Weak, but genuine -- a fair starting point")
print("for ML-06's signal audit and ML-08's actual model comparison, not the final answer.")

Note: you may need to restart the kernel to use updated packages.


Honest features, grouped split -- test AUC: 0.426
Base rate (future_decline): 39.1% -- this is the real, scaled, converged
baseline the honest feature set earns. Weak, but genuine -- a fair starting point
for ML-06's signal audit and ML-08's actual model comparison, not the final answer.


**Feature-importance sanity check.** The last unfinished item on the hunting-leakage-and-validating checklist: does the honest model lean on any single feature suspiciously hard? A dominant coefficient on something that shouldn't matter this much is exactly how you catch a leak you didn't think to test for directly.

In [3]:
# Re-fit on the same honest train/test split, but keep the fitted model and
# scaler this time (fit_scaled only returns predictions) so coefficients can
# actually be inspected -- coefficients are on comparable, standardized units
# since StandardScaler was fit before this, so magnitude is directly comparable.
scaler_check = StandardScaler().fit(X.iloc[train_idx])
model_check = LogisticRegression(max_iter=5000).fit(scaler_check.transform(X.iloc[train_idx]), y.iloc[train_idx])

coefs = pd.Series(model_check.coef_[0], index=honest_features).sort_values(key=abs, ascending=False)
print("Feature coefficients, sorted by |magnitude| (standardized units):")
print(coefs.round(3))

Feature coefficients, sorted by |magnitude| (standardized units):
char_count                 -1.416
word_count                  1.232
log_gsc_impressions         1.015
log_gsc_sum_position       -0.685
log_gsc_clicks             -0.487
gsc_avg_position            0.223
has_keyword_data           -0.200
log_scroll_events           0.145
has_word_count              0.140
has_ga4_data                0.125
days_since_last_update     -0.101
content_age_days            0.076
log_ga4_sessions           -0.073
has_backlink_data           0.070
category_count              0.054
days_with_impressions       0.054
prior_trend_pct             0.053
days_with_sessions          0.047
log_backlinks               0.037
has_position_data           0.022
ctr                         0.019
log_search_volume           0.017
scroll_rate                 0.015
engagement_rate             0.007
log_ga4_engaged_sessions    0.003
dtype: float64


**Verdict: no leak, but a real multicollinearity finding.** `char_count` (-1.416) and `word_count` (+1.232) are by far the two largest coefficients, with opposite signs — checked and confirmed: `word_count`/`char_count` correlate at **0.934** in `dim_content`. That's the classic signature of two near-redundant features fighting each other in a linear model (each absorbs part of the other's real signal, with unstable, inflated, opposite-sign coefficients), not a hidden leak — neither is future-derived or label-related. Nothing else here is remotely as dominant, and the honest AUC is already weak (0.426), so there's no "too good to be true" score to explain away. Worth a note for ML-08: tree-based models (what the reference pipeline actually compares) handle collinearity far better than logistic regression, so this may be a non-issue there — but if a linear model is ever used for real, consider dropping one of the two or using their ratio instead.

**Attack 1: inject the actual label-generating quantity.** `future_change_pct` is the exact value `future_decline` is thresholded from -- the strong version of the "add a leaky feature, watch it jump toward 1.0" test from the hunting-leakage-and-validating skill.

In [4]:
X_leaky1 = X.copy()
X_leaky1["future_change_pct"] = df["future_change_pct"].values
leaky1_probs = fit_scaled(X_leaky1.iloc[train_idx], y.iloc[train_idx], X_leaky1.iloc[test_idx])
leaky1_auc = roc_auc_score(y.iloc[test_idx], leaky1_probs)

print(f"WITH future_change_pct injected -- test AUC: {leaky1_auc:.3f}")
print(f"  jump from honest baseline: {leaky1_auc - honest_auc:+.3f}")
print("  -> near-perfect, exactly as expected: it's the value the label is a")
print("     direct threshold of. This is what a real leak looks like.")

WITH future_change_pct injected -- test AUC: 0.922
  jump from honest baseline: +0.496
  -> near-perfect, exactly as expected: it's the value the label is a
     direct threshold of. This is what a real leak looks like.


**Attack 2: a weaker, indirect leak.** `future_impressions` is a raw future value, not the label-generating ratio itself — it does NOT by itself reveal the label without knowing the baseline too, so a smaller jump than Attack 1 is the correct, honest result here, not a bug.

In [5]:
X_leaky2 = X.copy()
X_leaky2["future_impressions"] = df["future_impressions"].values
leaky2_probs = fit_scaled(X_leaky2.iloc[train_idx], y.iloc[train_idx], X_leaky2.iloc[test_idx])
leaky2_auc = roc_auc_score(y.iloc[test_idx], leaky2_probs)

print(f"WITH future_impressions injected -- test AUC: {leaky2_auc:.3f}")
print(f"  jump from honest baseline: {leaky2_auc - honest_auc:+.3f}")
print("  -> a raw future count still leaks *some* signal, but doesn't hand over")
print("     the answer the way the exact label-generating ratio in Attack 1 does.")

WITH future_impressions injected -- test AUC: 0.510
  jump from honest baseline: +0.084
  -> a raw future count still leaks *some* signal, but doesn't hand over
     the answer the way the exact label-generating ratio in Attack 1 does.


**Attack 3: random split vs. grouped split, honest features only.** Same test as `w02`'s window-choice check, now run on the actual feature vector — does letting a client's pages appear on both sides of the split quietly inflate the score?

In [6]:
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_probs = fit_scaled(X.iloc[train_idx_r], y.iloc[train_idx_r], X.iloc[test_idx_r])
random_auc = roc_auc_score(y.iloc[test_idx_r], random_probs)

print(f"Honest features, RANDOM split  -- test AUC: {random_auc:.3f}")
print(f"Honest features, GROUPED split -- test AUC: {honest_auc:.3f}")
print(f"  gap: {random_auc - honest_auc:+.3f} -- the random split's client leakage inflates the score")

Honest features, RANDOM split  -- test AUC: 0.594
Honest features, GROUPED split -- test AUC: 0.426
  gap: +0.169 -- the random split's client leakage inflates the score


**Timeline check.** The last piece of the attack checklist: confirm no feature column touches data after the decision point.

In [7]:
print("Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause")
print("in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.")

Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause
in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field(s) | Why |
|---|---|
| `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum`, `future_impressions` | The label itself, or computed strictly from the post-decision-point window. Confirmed in the leakage hunt above: injecting `future_change_pct` jumps AUC to 0.923; even the weaker `future_impressions` still leaks. |
| `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `sessions_referral`, `sessions_social`, `sessions_paid` | Traffic from channels (AI referral, paid, social, other-site referral) with no plausible link to this notebook's *GSC-impression-based* label — a different measurement system than what the target is built from. Extremely sparse too (85-99.7% zero), but that was the secondary reason, not the deciding one. |
| Every column in `fact_content_query_90d` | Its own window (`2026-04-02` to `2026-06-30`, verified in `w03`) overlaps this lane's label window — using any of its columns here would leak the future. |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing, and the sparsity + naming pattern strongly suggest these populate only when FlyRank's own system acted on a page — the "product decision as a feature" trap. Not proven safe, so excluded until independently verified. |
| `provider_used`, `model_used` | Explicitly marked "not a model feature" in the starter CSV's own data dictionary. |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero for every content item in this window (verified in `w03`) — zero variance, nothing to learn from in this slice. |
| `keyword_hash_id`, `url_hash_id`, `client_hash_id`, `content_hash_id`, `report_date`, `month` | Pseudonymous IDs / dates — grouping and windowing only, never model input. |
| `is_published`, `is_deleted` | Row filters (should exclude deleted/unpublished content before modeling), not signals to learn from. |
| Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, etc.) | Not shipped in this data — noted for completeness (`docs/ml-intern-dataset-and-lane-guide.md`, section 4). |
| Rows from clients with incomplete prior-window coverage | 18,652 rows (13.9%) dropped in section 1 — their `gsc_data_start` falls inside the 90-day prior window, so their "history" is partly a coverage gap, not real data. |

**Note on this exclusion, and why it doesn't touch `w03` (ML-04):** `w03_data_contract.ipynb` classified these seven columns as "too sparse to trust alone... ML-06 decides which survive" — that's still accurate; the data contract describes what the columns *are*, independent of what any downstream notebook chooses to use. This exclusion is a modeling decision made here in ML-05, not a correction to ML-04's classification.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.